# Loading ERA5

This notebook serves to extract, filter, and load ERA5 data from Google's public ERA5 analysis-ready, cloud-optimised (ARCO) mirror into Azure Blob Storage. Here, ERA5 data for the 1940-01-01 to 2025-12-31 period (continually, if irregularly, updated) at hourly frequency is stored in Zarr format. Beyond format, the sole difference between re-gridded data available therein and that through the Copernicus Climate Data Store (CDS) is variable naming: longnames are used in the former and shortnames in the latter.

See the [GCP ERA5 ARCO bucket](https://console.cloud.google.com/storage/browser/gcp-public-data-arco-era5) for other datasets including alternatively gridded Zarr and raw source NetCDF files. Not all datasets contain every variable or the same time range / frequency.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import xarray as xr  # also requires zarr, fsspec, gcsfs, dask

# insert parent directory to path for proper absolute local imports
sys.path.insert(0, str(Path.cwd().parent.parent.resolve()))
from setup.common.constants import (
    END_DATETIME,
    FREQUENCY,
    GCP_ERA5_URI,
    LOCAL_DATA_PATH,
    START_DATETIME,
)
from setup.components.common.constants import (
    ATMOS_LEVELS,
    ATMOS_VAR_MAP,
    STATIC_VAR_MAP,
    SURF_VAR_MAP,
)

Define the surface and pressure level variables and pressure levels to load. Variable longnames are mapped to shortnames for convenience (particularly when reading data into Aurora `Batch` objects) and are non-functional.

To ingest new variables and levels, add the former by longname to the appropriate dictionary and the latter by integer pressure level (hPa) to the given list.

**NOTE:** The two variable mappings are not strict. That is, single-level variables can be added to the pressure level variable mapping without error, they simply afford separation and readability. For the minimum set of variables required for Aurora 0.25 pre-trained, see `setup.components.common.constants`.

In [ ]:
EXTRA_SFC_VARS = {
    "2m_dewpoint_temperature": "d2m",
}
EXTRA_ATMOS_VARS = {}
EXTRA_LEVELS = []

Lazy load and subset data by variables, levels, time range, and timestep.

**NOTE:** This will take at least 1 minute regardless of subset size due to the need to load metadata for this PB scale dataset.

In [ ]:
ds = xr.open_zarr(GCP_ERA5_URI, chunks={})  # type: ignore[arg-type]
variables = [
    *SURF_VAR_MAP.keys(),
    *STATIC_VAR_MAP.keys(),
    *ATMOS_VAR_MAP.keys(),
    *EXTRA_SFC_VARS.keys(),
    *EXTRA_ATMOS_VARS.keys(),
]
var_subset_ds = ds[variables]
# adjust start to -6h for initial state being t0 and t-6h
adjusted_start = np.datetime64(START_DATETIME) - np.timedelta64(6, "h")
subset_ds = var_subset_ds.sel(
    time=slice(adjusted_start, np.datetime64(END_DATETIME), FREQUENCY),
    level=ATMOS_LEVELS + EXTRA_LEVELS,
)
# update metadata attributes to reflect the subset, not original, data
subset_ds.attrs.update(valid_time_start=START_DATETIME, valid_time_stop=END_DATETIME)
subset_ds

Write subset data to local storage.

In [ ]:
subset_ds.to_zarr(
    LOCAL_DATA_PATH,
    mode="w",
    compute=True,
    consolidated=True,
    zarr_format=2,
)
print(f"Wrote to: {LOCAL_DATA_PATH}")

Confirm persisted data is available and valid.

**NOTE:** An equality check with the original `subset_ds` (e.g. `new_ds.equals(subset_ds)`) can be used for the avoidance of doubt but requires loading data into memory, which may take time and result in an OOM error, subset size dependent.

In [ ]:
xr.open_dataset(LOCAL_DATA_PATH, engine="zarr", chunks={})  # type: ignore[arg-type]